In [1]:
from pathlib import Path
from decouple import AutoConfig

from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_classic.agents import AgentExecutor, initialize_agent, create_react_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_openai import ChatOpenAI


/var/folders/f5/jv7k5yw97479jf4xr73rgvd40000gn/T/ipykernel_93947/494961353.py:9: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools.python.tool import PythonREPLTool


In [2]:
cwd = Path.cwd()
project_dir = cwd / "langchain" if (cwd / "langchain").is_dir() else cwd
config = AutoConfig(search_path=str(project_dir))
api_key = config("OPENAI_API_KEY")

In [3]:
openai = ChatOpenAI(
    model_name="gpt-4o-mini",
    openai_api_key=api_key,
    temperature=0,
    store=False
)

In [4]:
prompt = """
Como assinstente financeiro pessoal, ajude a responder as seguintes perguntas com ajuda da internet.
Perguntas: {q}
"""
prompt_template = PromptTemplate.from_template(prompt)
print(prompt_template)

input_variables=['q'] input_types={} partial_variables={} template='\nComo assinstente financeiro pessoal, ajude a responder as seguintes perguntas com ajuda da internet.\nPerguntas: {q}\n'


In [6]:
react_prompt = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""
react_instructions = PromptTemplate.from_template(react_prompt)
print(react_instructions)


input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}\n'


In [7]:
python_repl = PythonREPLTool()
python_repl_tool = Tool(
    name='Python REPL',
    func=python_repl.run,
    description="""
    Qualquer tipo de calculo deve usar esta ferramenta. Voce nao deve realizar o calculo diretamente. Voce deve inserir codigo Python.
    """
)

In [8]:
search = DuckDuckGoSearchRun()
ddg_tool = Tool(
    name='Busca DuckDuckGo',
    func=search.run,
    description="""
    Util para encontrar informacoes e dicas de economia e opcoes de investimento.
    Voce sempre deve pesquisar na internet as melhores dicas usando esta ferramenta, nao
    responda diretamente. Sua resposta deve informar que ha elementos pesquisados na internet.
    """
)

In [10]:
tools = [python_repl_tool, ddg_tool]
agent = create_react_agent(
    openai,
    tools,
    react_instructions
)

In [22]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_error=True,
    max_iterations=10
)
print(agent_executor)

verbose=True agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': "Python REPL(tool_input: 'str | dict[str, Any]', verbose: 'bool | None' = None, start_color: 'str | None' = 'green', color: 'str | None' = 'green', callbacks: 'Callbacks' = None, *, tags: 'list[str] | None' = None, metadata: 'dict[str, Any] | None' = None, run_name: 'str | None' = None, run_id: 'uuid.UUID | None' = None, config: 'RunnableConfig | None' = None, tool_call_id: 'str | None' = None, **kwargs: 'Any') -> 'Any' - \n    Qualquer tipo de calculo deve usar esta ferramenta. Voce nao deve realizar o calculo diretamente. Voce deve inserir codigo Python.\n    \nBusca DuckDuckGo(tool_input: 'str | dict[str, Any]', verbose: 'bool | None' = None, start_color: 'str | None' = 'green', color: 'str | None' = 'green', callbacks: '

In [23]:
question = """
Eu ganho R$ 4000 por mes mas o total de minhas dispesas é R$3800 mais 500 de aluguel.
Como posso ajustar meu orçamento para economiar dinheiro?
"""

In [24]:
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...
Para ajudar a responder a pergunta sobre como ajustar o orçamento para economizar dinheiro, primeiro, preciso entender melhor a situação financeira. Você mencionou que ganha R$ 4000 por mês e tem despesas de R$ 3800, além de R$ 500 de aluguel. Isso significa que suas despesas totais são de R$ 4300, o que resulta em um déficit mensal de R$ 300.

Para encontrar dicas sobre como ajustar o orçamento e economizar dinheiro, vou pesquisar na internet. 

Action: Busca DuckDuckGo  
Action Input: "como ajustar orçamento pessoal para economizar dinheiro"  Mar 3, 2025 · Como gastar, economizar e distribuir seu dinheiro com sabedoria? Neste artigo, você encontrará instruções passo a passo e conselhos de especialistas que ajudarão você a gerenciar efetivamente suas finanças pessoais e criar uma rede de segurança confiável. Aprenda a criar um orçamento pessoal realista, controlar gastos, organizar contas, formar reservas e alcançar metas financeiras sem abrir m

In [25]:
print(output['input'])


Como assinstente financeiro pessoal, ajude a responder as seguintes perguntas com ajuda da internet.
Perguntas: 
Eu ganho R$ 4000 por mes mas o total de minhas dispesas é R$3800 mais 500 de aluguel.
Como posso ajustar meu orçamento para economiar dinheiro?




In [26]:
print(output['output'])

Para ajustar seu orçamento e economizar dinheiro, considere as seguintes ações:

1. **Reveja suas despesas**: Analise suas despesas mensais de R$ 3800 e veja onde pode cortar. Por exemplo, considere reduzir gastos com lazer, alimentação ou serviços.
2. **Estabeleça um limite de gastos**: Defina um limite para cada categoria de despesa e tente não ultrapassá-lo.
3. **Considere alternativas de moradia**: Se o aluguel de R$ 500 for uma parte significativa de suas despesas, veja se há opções mais baratas ou se você pode dividir o aluguel com alguém.
4. **Crie um fundo de emergência**: Mesmo que você esteja no vermelho, tente reservar uma pequena quantia todo mês para um fundo de emergência.
5. **Busque fontes de renda extra**: Considere trabalhos freelance ou outras formas de aumentar sua renda.

Essas estratégias podem ajudá-lo a equilibrar seu orçamento e economizar dinheiro a longo prazo.


In [27]:
question = """
Minha renda é de R$ 6000 por mes, o total de minhas despesas é de R$ 2500 mais 1000 de aluguel.
Quais dicas de investimento me da?
"""

In [28]:
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...
Para responder a essa pergunta, preciso buscar informações sobre dicas de investimento que sejam adequadas para alguém com uma renda mensal de R$ 6000 e despesas totais de R$ 3500 (R$ 2500 + R$ 1000 de aluguel). Vou procurar por sugestões que possam ajudar a maximizar o uso do dinheiro disponível.

Action: Busca DuckDuckGo  
Action Input: "dicas de investimento para renda de R$ 6000 e despesas de R$ 3500"  Nov 22, 2025 · Descubra qual investimento rende mais e conheça opções de renda fixa, CDBs, Tesouro Direto e fundos que equilibram rentabilidade e segurança. May 21, 2026 · As Letras de Crédito Imobiliário (LCI) e do Agronegócio (LCA) são títulos de renda fixa isentos de Imposto de Renda (IR) para pessoas físicas. São opções de investimento recomendadas para o médio prazo porque geralmente possuem um período de carência. Jun 17, 2026 · A ferramenta mostra quanto cada tipo de investimento pode render no período escolhido. Desta forma, você conseg

In [29]:
print(output['input'])


Como assinstente financeiro pessoal, ajude a responder as seguintes perguntas com ajuda da internet.
Perguntas: 
Minha renda é de R$ 6000 por mes, o total de minhas despesas é de R$ 2500 mais 1000 de aluguel.
Quais dicas de investimento me da?




In [21]:
print(output['output'])

Para alguém com uma renda mensal de R$ 6000 e despesas totais de R$ 3500, aqui estão algumas dicas de investimento:

1. **Reserva de Emergência**: Antes de começar a investir, é importante ter uma reserva de emergência. Isso deve cobrir de 3 a 6 meses de despesas. Considere colocar esse valor em uma conta de poupança ou em um investimento de alta liquidez.

2. **Renda Fixa**: Considere investir em produtos de renda fixa, como CDBs (Certificados de Depósito Bancário), Tesouro Direto e Letras de Crédito (LCI e LCA). Esses investimentos são mais seguros e oferecem rendimentos previsíveis.

3. **Fundos de Investimento**: Explore fundos que equilibram rentabilidade e segurança. Existem fundos de renda fixa e multimercados que podem ser adequados para o seu perfil.

4. **Investimentos em Ações**: Se você estiver disposto a assumir um pouco mais de risco, considere investir uma parte do seu capital em ações ou ETFs (fundos de índice). Isso pode oferecer um potencial de retorno maior a longo p